# ETL — Índice HHI de Concentração de Mercado
**TCC — Preços e concentração de mercado em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

**O que este notebook faz:**  
Calcula o Índice de Herfindahl-Hirschman (HHI) por item e ano,
medindo a concentração de fornecedores no mercado de insumos hospitalares.

**Período coberto:** 2009–2025 (17 anos · todos os anos com CNPJ disponível)

**Classificação adotada (CADE / FTC Internacional):**

| HHI | Classificação |
|-----|---------------|
| < 1.500 | Não Concentrado |
| 1.500 – 2.500 | Moderadamente Concentrado |
| > 2.500 | Altamente Concentrado |

**Nota metodológica:**  
O HHI é calculado pelo valor financeiro total de cada fornecedor (CNPJ) por item (CATMAT) e ano,
refletindo dominância de receita no mercado — abordagem padrão para compras públicas.
Os arquivos de 2000–2008 foram excluídos por ausência de padronização do CATMAT e CNPJ.

## 1. Montar o Google Drive

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────


Mounted at /content/drive


## 2. Instalar dependências

In [2]:
!pip install polars pyarrow --quiet
print('Dependências instaladas ✓')

Dependências instaladas ✓


## 3. Configuração
> ⚙️ Ajuste os caminhos abaixo.

In [3]:
import polars as pl
import gc
from pathlib import Path

# ── AJUSTE AQUI ──────────────────────────────────────────────────────────
PASTA_TRATADOS = Path(Path(PASTA_DADOS) / 'Base Harmonizacao Campos')
PASTA_HHI      = Path(Path(PASTA_DADOS) / 'Base HHI')
# ─────────────────────────────────────────────────────────────────────────

PASTA_HHI.mkdir(parents=True, exist_ok=True)

SAIDA_PARQUET = PASTA_HHI / 'hhi_concentracao.parquet'
SAIDA_CSV     = PASTA_HHI / 'hhi_concentracao.csv'

# Confirma arquivos disponíveis
parquets_bps = sorted(PASTA_TRATADOS.glob('BPS_*.parquet'))
parquets_hhi = [p for p in parquets_bps if 2009 <= int(p.stem.split('_')[1]) <= 2023]
print(f'Total de parquets BPS  : {len(parquets_bps)}')
print(f'Parquets para o HHI    : {len(parquets_hhi)} (2009–2023)')
print(f'Pasta saída            : {PASTA_HHI}')

Total de parquets BPS  : 26
Parquets para o HHI    : 17 (2009–2025)
Pasta saída            : /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base HHI


## 4. Verificar Colunas Disponíveis
> Confirma que todos os arquivos de 2009–2025 têm `cod_catmat` e `cnpj_fornecedor` e `valor_total`.

In [4]:
print('Verificando colunas por ano...')
problemas = []
for p in parquets_hhi:
    cols = pl.read_parquet(p, n_rows=0).columns
    faltando = [c for c in ['cod_catmat', 'cnpj_fornecedor', 'valor_total'] if c not in cols]
    if faltando:
        problemas.append(p.name)
        print(f'  {p.name}: PROBLEMA — faltando {faltando}')
        print(f'         colunas presentes: {cols}')
    else:
        print(f'  {p.name}: OK ✓')

if problemas:
    print(f'\n⚠ Arquivos com problema: {problemas}')
    print('Verifique se o campo está com rótulo trocado (como em 2018–2020) antes de excluir o ano.')
else:
    print('\nTodos os arquivos prontos para o cálculo do HHI ✓')

Verificando colunas por ano...
  BPS_2009.parquet: OK ✓
  BPS_2010.parquet: OK ✓
  BPS_2011.parquet: OK ✓
  BPS_2012.parquet: OK ✓
  BPS_2013.parquet: OK ✓
  BPS_2014.parquet: OK ✓
  BPS_2015.parquet: OK ✓
  BPS_2016.parquet: OK ✓
  BPS_2017.parquet: OK ✓
  BPS_2018.parquet: OK ✓
  BPS_2019.parquet: OK ✓
  BPS_2020.parquet: OK ✓
  BPS_2021.parquet: OK ✓
  BPS_2022.parquet: OK ✓
  BPS_2023.parquet: OK ✓
  BPS_2024.parquet: OK ✓
  BPS_2025.parquet: OK ✓

Todos os arquivos prontos para o cálculo do HHI ✓


Check Claude

In [5]:
import polars as pl
from pathlib import Path

PASTA = Path(Path(PASTA_DADOS) / 'Base Harmonizacao Campos')

for ano in [2009, 2010, 2011, 2012, 2013]:
    df = pl.read_parquet(PASTA / f'BPS_{ano}.parquet',
                         columns=['cod_catmat', 'cnpj_fornecedor', 'valor_total'])
    total = df.height
    catmat_ok = df.filter(pl.col('cod_catmat').cast(pl.Utf8).str.strip_chars().str.len_chars() > 0).height
    cnpj_ok   = df.filter(
        (pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars().str.len_chars() > 0) &
        (pl.col('cnpj_fornecedor').cast(pl.Utf8) != '00000000000000')
    ).height
    valor_ok  = df.filter(pl.col('valor_total') > 0).height
    print(f'BPS_{ano}: {total:>7,} linhas | CATMAT preenchido: {catmat_ok:>7,} | CNPJ válido: {cnpj_ok:>7,} | valor>0: {valor_ok:>7,}')
    print(f'   exemplos cnpj_fornecedor: {df["cnpj_fornecedor"].head(3).to_list()}')
    print(f'   exemplos cod_catmat:      {df["cod_catmat"].head(3).to_list()}\n')

BPS_2009:  13,494 linhas | CATMAT preenchido:  13,494 | CNPJ válido:  13,494 | valor>0:  13,494
   exemplos cnpj_fornecedor: ['57507378000101', '03474341000197', '03951140000133']
   exemplos cod_catmat:      ['BR0273400', 'BR0268124', 'BR0302457']

BPS_2010:  19,069 linhas | CATMAT preenchido:  19,069 | CNPJ válido:  19,069 | valor>0:  19,067
   exemplos cnpj_fornecedor: ['33228701000131', '44734671000151', '44734671000151']
   exemplos cod_catmat:      ['BR0267731', 'BR0267660', 'BR0267635']

BPS_2011:  13,611 linhas | CATMAT preenchido:  13,611 | CNPJ válido:  13,611 | valor>0:  13,611
   exemplos cnpj_fornecedor: ['67729178000491', '67729178000491', '08432330000168']
   exemplos cod_catmat:      ['BR0270968', 'BR0267502', 'BR0267772']

BPS_2012:   4,875 linhas | CATMAT preenchido:   4,875 | CNPJ válido:   4,875 | valor>0:   4,875
   exemplos cnpj_fornecedor: ['07847837000110', '29976032000109', '07847837000110']
   exemplos cod_catmat:      ['BR0279278', 'BR0348265', 'BR0279561']



In [6]:
import polars as pl
from pathlib import Path

PASTA = Path(Path(PASTA_DADOS) / 'Base Harmonizacao Campos')

for ano in [2009, 2013]:  # um problemático e um bom, para comparar
    df = pl.read_parquet(PASTA / f'BPS_{ano}.parquet')
    print(f'═══ BPS_{ano} ═══')
    print(f'  colunas: {df.columns}\n')
    # mostra valor_total e os campos que poderiam recalculá-lo
    cols_interesse = [c for c in ['valor_total','preco_unitario','quantidade'] if c in df.columns]
    print(df.select(cols_interesse).head(5))
    print(f'  dtype de cada: {[(c, df[c].dtype) for c in cols_interesse]}\n')

═══ BPS_2009 ═══
  colunas: ['data_compra', 'cod_catmat', 'desc_item', 'unidade_fornecimento', 'preco_unitario', 'quantidade', 'modalidade_compra', 'tipo_compra', 'uf', 'municipio', 'instituicao', 'cnpj_instituicao', 'fornecedor', 'cnpj_fornecedor', 'fabricante', 'cnpj_fabricante', 'registro_anvisa', 'generico', 'esfera', 'ano', 'unidade_chave', 'unidade_rotulo', 'valor_total']

shape: (5, 3)
┌─────────────┬────────────────┬────────────┐
│ valor_total ┆ preco_unitario ┆ quantidade │
│ ---         ┆ ---            ┆ ---        │
│ f64         ┆ f64            ┆ f64        │
╞═════════════╪════════════════╪════════════╡
│ 356.0       ┆ 0.089          ┆ 4000.0     │
│ 1400.0      ┆ 0.2            ┆ 7000.0     │
│ 2120.58     ┆ 1.62           ┆ 1309.0     │
│ 18400.0     ┆ 1.84           ┆ 10000.0    │
│ 264.0       ┆ 0.11           ┆ 2400.0     │
└─────────────┴────────────────┴────────────┘
  dtype de cada: [('valor_total', Float64), ('preco_unitario', Float64), ('quantidade', Float64)]


In [7]:
import polars as pl
from pathlib import Path

PASTA_CSV = Path(Path(PASTA_DADOS) / 'Base Conversao CSV')

# lê só as primeiras linhas como TEXTO PURO, sem conversão nenhuma
df = pl.read_csv(PASTA_CSV / 'BPS_2009.csv', infer_schema_length=0)  # tudo como string
print('Colunas do CSV bruto 2009:')
print(df.columns)
print()

# tenta achar as colunas de preço/valor pelo nome original
import re
cols_preco = [c for c in df.columns if re.search(r'unit|pre[çc]o|valor|total|pago', c, re.I)]
print(f'Colunas candidatas a preço/valor: {cols_preco}\n')

for c in cols_preco:
    print(f'  "{c}" → exemplos: {df[c].head(5).to_list()}')

Colunas do CSV bruto 2009:
['\xa0Código BR\xa0', '\xa0Descrição Item\xa0', '\xa0Unidade de Fornecimento\xa0', '\xa0Genérico\xa0', '\xa0Registro Anvisa\xa0', '\xa0Qtd Itens Comprados\xa0', '\xa0Preço Unitário\xa0', '\xa0Data Compra\xa0', '\xa0Modalidade de Compra\xa0', '\xa0Data Inserção\xa0', '\xa0Tipo Compra\xa0', '\xa0Fabricante\xa0', '\xa0CNPJ Fabricante\xa0', '\xa0Fornecedor\xa0', '\xa0CNPJ Fornecedor\xa0', '\xa0Nome Instituição\xa0', '\xa0CNPJ Instituição\xa0', '\xa0Município\xa0', '\xa0Esfera\xa0', '\xa0UF\xa0', '\xa0Licitação\xa0', '\xa0Nota Fiscal\xa0']

Colunas candidatas a preço/valor: ['\xa0Preço Unitário\xa0']

  " Preço Unitário " → exemplos: ['\xa0 0,0890\xa0', '\xa0 0,2000\xa0', '\xa0 1,6200\xa0', '\xa0 1,8400\xa0', '\xa0 0,1100\xa0']


## 5. Calcular o HHI
> **Correção SchemaError:** lê cada arquivo individualmente selecionando
> apenas as 4 colunas necessárias antes de concatenar.
> Isso evita o erro quando anos diferentes têm colunas extras (ex: `classe_catmat` em 2018–2020).

In [8]:
print('Iniciando cálculo do HHI...')

# ── LEITURA COM SCHEMA UNIFORME ──────────────────────────────────────────
# Lemos cada arquivo separadamente, pinçando só as 4 colunas que importam.
# Isso garante que diferenças de schema entre anos não causem SchemaError.
COLUNAS_HHI = ['ano', 'data_compra', 'cod_catmat', 'cnpj_fornecedor', 'valor_total']

lista_lazy = []
for p in parquets_hhi:
    cols_disponiveis = pl.read_parquet(p, n_rows=0).columns
    cols_presentes   = [c for c in COLUNAS_HHI if c in cols_disponiveis]
    if len(cols_presentes) == len(COLUNAS_HHI):
        lista_lazy.append(pl.scan_parquet(p).select(COLUNAS_HHI))
    else:
        faltando = [c for c in COLUNAS_HHI if c not in cols_disponiveis]
        print(f'  ⚠ {p.name} ignorado — colunas ausentes: {faltando}')

print(f'Arquivos incluídos no cálculo: {len(lista_lazy)}')

# Concatena com schema uniforme
lf_bps = pl.concat(lista_lazy)

# ── PIPELINE DE CÁLCULO ──────────────────────────────────────────────────
lf_hhi = (
    lf_bps

    # Filtra registros inválidos
    .filter(
        # ── RECORTE TEMPORAL 2009-2023 (decidido em set/2026)
        # O ano vem de data_compra, nao do nome do arquivo. 2024 esta incompleto
        # na fonte (compras ate 31/10) e 2025 foi descartado por vir de extracao
        # do painel, com semantica de data distinta (homologacao, nao compra).
        pl.col('data_compra').is_not_null() &
        pl.col('data_compra').dt.year().is_between(2009, 2023) &
        pl.col('cod_catmat').is_not_null() &
        pl.col('cod_catmat').cast(pl.Utf8).str.strip_chars().str.len_chars().gt(0) &
        pl.col('cnpj_fornecedor').is_not_null() &
        pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars().str.len_chars().gt(0) &
        pl.col('cnpj_fornecedor').cast(pl.Utf8).ne('00000000000000') &
        pl.col('valor_total').is_not_null() &
        pl.col('valor_total').gt(0)
    )
    # ano derivado da data da compra, substituindo o ano do nome do arquivo
    .with_columns(pl.col('data_compra').dt.year().cast(pl.Int32).alias('ano'))

    # Passo 1: valor de cada fornecedor por item e ano
    .group_by(['ano', 'cod_catmat', 'cnpj_fornecedor'])
    .agg(pl.col('valor_total').sum().alias('valor_fornecedor'))

    # Passo 2: valor total do mercado (mesmo item, mesmo ano)
    .with_columns(
        pl.col('valor_fornecedor')
        .sum().over(['ano', 'cod_catmat'])
        .alias('valor_mercado_total')
    )

    # Passo 3: market share (%) ao quadrado
    .with_columns(
        ((pl.col('valor_fornecedor') / pl.col('valor_mercado_total')) * 100)
        .alias('market_share')
    )
    .with_columns(
        (pl.col('market_share') ** 2).alias('market_share_sq')
    )

    # Passo 4: HHI = soma dos quadrados por item e ano
    .group_by(['ano', 'cod_catmat'])
    .agg(
        pl.col('market_share_sq').sum().alias('hhi'),
        pl.col('cnpj_fornecedor').n_unique().alias('qtd_fornecedores'),
        pl.col('valor_mercado_total').first().alias('volume_financeiro')
    )

    # Passo 5: classificação de mercado (CADE / FTC)
    .with_columns(
        pl.when(pl.col('hhi') < 1500)
        .then(pl.lit('Não Concentrado'))
        .when(pl.col('hhi') < 2500)
        .then(pl.lit('Moderadamente Concentrado'))
        .otherwise(pl.lit('Altamente Concentrado'))
        .alias('classificacao_mercado')
    )

    .sort(['ano', 'cod_catmat'])
)

# Grava Parquet sem materializar tudo na RAM (sink = streaming)
print('Gravando hhi_concentracao.parquet...')
lf_hhi.sink_parquet(SAIDA_PARQUET)
print('Parquet gravado ✓')

Iniciando cálculo do HHI...
Arquivos incluídos no cálculo: 17
Gravando hhi_concentracao.parquet...
Parquet gravado ✓


## 6. Exportar CSV e Validar

In [9]:
df_hhi = pl.read_parquet(SAIDA_PARQUET)

# Exporta CSV
df_hhi.write_csv(SAIDA_CSV)
print(f'CSV gravado ✓')

# ── Estatísticas
print(f'\n=== RESULTADO ===')
print(f'Combinações item-ano analisadas: {df_hhi.shape[0]:,}')
print(f'Colunas                        : {df_hhi.columns}')
print(f'Anos cobertos                  : {sorted(df_hhi["ano"].unique().to_list())}')

print(f'\nDistribuição por classificação:')
print(
    df_hhi.group_by('classificacao_mercado')
    .agg(pl.len().alias('qtd'))
    .sort('qtd', descending=True)
)

print(f'\nTop 10 mercados mais concentrados (HHI alto):')
print(
    df_hhi.sort('hhi', descending=True)
    .select(['ano','cod_catmat','hhi','qtd_fornecedores','classificacao_mercado'])
    .head(10)
)

print(f'\nTop 10 mercados mais competitivos (mín. 5 fornecedores):')
print(
    df_hhi.filter(pl.col('qtd_fornecedores') >= 5)
    .sort('hhi')
    .select(['ano','cod_catmat','hhi','qtd_fornecedores','classificacao_mercado'])
    .head(10)
)

del df_hhi
gc.collect()
print(f'\nArquivos em: {PASTA_HHI}')
print(f'  • hhi_concentracao.parquet')
print(f'  • hhi_concentracao.csv')

CSV gravado ✓

=== RESULTADO ===
Combinações item-ano analisadas: 81,021
Colunas                        : ['ano', 'cod_catmat', 'hhi', 'qtd_fornecedores', 'volume_financeiro', 'classificacao_mercado']
Anos cobertos                  : [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Distribuição por classificação:
shape: (3, 2)
┌───────────────────────────┬───────┐
│ classificacao_mercado     ┆ qtd   │
│ ---                       ┆ ---   │
│ str                       ┆ u32   │
╞═══════════════════════════╪═══════╡
│ Altamente Concentrado     ┆ 73132 │
│ Moderadamente Concentrado ┆ 5355  │
│ Altamente Competitivo     ┆ 2534  │
└───────────────────────────┴───────┘

Top 10 mercados mais concentrados (HHI alto):
shape: (10, 5)
┌──────┬────────────┬─────────┬──────────────────┬───────────────────────┐
│ ano  ┆ cod_catmat ┆ hhi     ┆ qtd_fornecedores ┆ classificacao_mercado │
│ ---  ┆ ---        ┆ ---     ┆ ---              ┆ ---        

## 7. Texto para o Material e Métodos

Cole este trecho no seu TCC:

> *O Índice de Herfindahl-Hirschman (HHI) foi calculado por item (código CATMAT)
> e por ano, com base no valor financeiro total transacionado por cada fornecedor
> (CNPJ) no período de 2009 a 2025. O período anterior a 2009 foi excluído por
> ausência de padronização do código CATMAT e do CNPJ do fornecedor nos registros
> do BPS. A classificação de mercado seguiu os critérios do CADE e do Federal Trade
> Commission (FTC): HHI inferior a 1.500 indica mercado altamente competitivo;
> entre 1.500 e 2.500, moderadamente concentrado; e acima de 2.500, altamente
> concentrado (FTC, 2010). O índice foi calculado a nível nacional por ano,
> constituindo uma análise macro da estrutura de mercado dos insumos hospitalares
> públicos brasileiros ao longo de 17 anos.*

**Referência para incluir nas Referências do TCC:**

> FEDERAL TRADE COMMISSION (FTC). *Horizontal Merger Guidelines*. Washington: FTC/DOJ, 2010.
> Disponível em: https://www.ftc.gov/sites/default/files/attachments/merger-review/100819hmg.pdf

## Memória de cálculo da seção de concentração
> Reproduz, com a conta explicita, cada valor citado no texto do TCC. O objetivo
> e permitir que qualquer numero publicado seja refeito e conferido.
>
> Valores que NAO sao calculados aqui, por serem citados de fonte externa:
> as faixas de 1.500 e 2.500 pontos, que vem do Guia do CADE (CADE, 2016).

In [ ]:
import polars as pl
from pathlib import Path
BASE = Path(PASTA_DADOS)
h = pl.read_parquet(BASE / 'Base HHI' / 'hhi_concentracao.parquet')
if 'qtd_fornecedores' in h.columns:
    h = h.rename({'qtd_fornecedores': 'n_forn'})
TOT = h.height
print('=' * 74)
print('1. DISTRIBUICAO POR FAIXA DE CONCENTRACAO')
print('=' * 74)
print(f'{"faixa":<28} | {"mercados":>9} | {"conta":<22} | {"%":>6}')
for faixa in ['Altamente Concentrado','Moderadamente Concentrado','Não Concentrado']:
    n = h.filter(pl.col('classificacao_mercado') == faixa).height
    print(f'{faixa:<28} | {n:>9,} | {n:>7,} / {TOT:,} = | {n/TOT:>5.1%}')
print(f'{"TOTAL":<28} | {TOT:>9,} |')

In [ ]:
print('=' * 74)
print('2. FORNECEDORES POR MERCADO')
print('=' * 74)
med = h['n_forn'].median()
unico = h.filter(pl.col('n_forn') == 1).height
print(f'  mediana de fornecedores distintos por mercado : {med:.0f}')
print(f'  mercados com UM unico fornecedor              : {unico:,} / {TOT:,} = {unico/TOT:.1%}')
print(f'  indice mediano da serie (HHI)                 : {h["hhi"].median():,.0f} pontos')
print(f'    -> compare com o limiar de 2.500 do CADE: {h["hhi"].median()/2500:.1f}x')

In [ ]:
print('=' * 74)
print('3. RESTRICOES PROGRESSIVAS — a concentracao resiste?')
print('=' * 74)
print(f'{"restricao":<26} | {"mercados":>9} | {"alt.concentrados":>17} | {"%":>6} | {"HHI mediano":>11}')
for rot, k in [('sem restricao', 1), ('2 ou mais fornecedores', 2),
               ('5 ou mais fornecedores', 5), ('10 ou mais fornecedores', 10)]:
    s = h.filter(pl.col('n_forn') >= k)
    a = s.filter(pl.col('classificacao_mercado') == 'Altamente Concentrado').height
    print(f'{rot:<26} | {s.height:>9,} | {a:>17,} | {a/s.height:>5.1%} | {s["hhi"].median():>11,.0f}')
print('\nLeitura: mesmo excluindo mercados de baixa frequencia de compra, a maioria')
print('permanece concentrada — a concentracao nao e artefato de registros esparsos.')

In [ ]:
print('=' * 74)
print('4. CONCENTRACAO PONDERADA PELO VALOR TRANSACIONADO')
print('=' * 74)
col_vol = 'volume_financeiro' if 'volume_financeiro' in h.columns else 'volume_financeiro_mercado'
VT = h[col_vol].sum()
for rot, filtro in [('altamente concentrados', pl.col('classificacao_mercado') == 'Altamente Concentrado'),
                    ('de fornecedor unico',    pl.col('n_forn') == 1)]:
    s = h.filter(filtro)
    print(f'  mercados {rot:<24}: {s.height/TOT:>5.1%} dos mercados | '
          f'{s[col_vol].sum()/VT:>5.1%} do valor')
print('\nLeitura: a concentracao nao se limita a itens de baixa expressao financeira.')

In [ ]:
print('=' * 74)
print('5. EVOLUCAO ANUAL E COBERTURA DA BASE')
print('=' * 74)
ev = (h.group_by('ano').agg(pl.len().alias('mercados'),
                            pl.col('hhi').median().round(0).alias('hhi_mediano'),
                            (pl.col('classificacao_mercado')=='Altamente Concentrado').mean().alias('p_alt'),
                            (pl.col('n_forn')==1).mean().alias('p_unico')).sort('ano'))
print(f'{"ano":>5} | {"mercados":>9} | {"HHI mediano":>11} | {"% alt.conc":>10} | {"% unico":>8}')
for r in ev.iter_rows(named=True):
    print(f'{r["ano"]:>5} | {r["mercados"]:>9,} | {r["hhi_mediano"]:>11,.0f} | '
          f'{r["p_alt"]:>9.1%} | {r["p_unico"]:>7.1%}')
print('\nATENCAO METODOLOGICA: o ano de menor concentracao coincide com o de maior')
print('cobertura da base. A coincidencia e OBSERVADA, nao testada — nao se mediu se')
print('a variacao do indice decorre da ampliacao do registro ou de alteracao real')
print('na estrutura de fornecimento. O texto declara essa limitacao.')
print(f'\nExtremos de cobertura: {ev["mercados"].min():,} mercados em '
      f'{ev.sort("mercados")["ano"][0]} e {ev["mercados"].max():,} em {ev.sort("mercados")["ano"][-1]}')

In [ ]:
print('=' * 74)
print('6. DISPERSAO ENTRE MERCADOS')
print('=' * 74)
comp = h.filter(pl.col('n_forn') >= 5).sort('hhi').head(1)
print(f'  maximo de fornecedores em um mercado : {h["n_forn"].max()}')
print(f'  mercado menos concentrado (min. 5 forn.): {comp["cod_catmat"][0]} em {comp["ano"][0]}')
print(f'    HHI = {comp["hhi"][0]:,.0f} pontos com {comp["n_forn"][0]} fornecedores')

# todos os valores como float: a coluna mistura contagem e percentual, e o
# Polars infere o tipo pelo primeiro elemento — sem o cast, a gravacao falha.
valores = [float(TOT),
           round(h.filter(pl.col('classificacao_mercado')=='Altamente Concentrado').height/TOT*100, 1),
           round(unico/TOT*100, 1),
           round(float(h['hhi'].median()), 0),
           float(med),
           *[round(h.filter(pl.col('n_forn')>=k)
                    .filter(pl.col('classificacao_mercado')=='Altamente Concentrado').height
                   / h.filter(pl.col('n_forn')>=k).height*100, 1) for k in (2, 5, 10)],
           round(h.filter(pl.col('classificacao_mercado')=='Altamente Concentrado')[col_vol].sum()/VT*100, 1),
           round(h.filter(pl.col('n_forn')==1)[col_vol].sum()/VT*100, 1)]
resumo = pl.DataFrame({
    'indicador': ['mercados_item_ano','pct_altamente_concentrado','pct_fornecedor_unico',
                  'hhi_mediano','mediana_fornecedores','pct_alt_conc_min2','pct_alt_conc_min5',
                  'pct_alt_conc_min10','pct_valor_alt_conc','pct_valor_forn_unico'],
    'valor': pl.Series(valores, dtype=pl.Float64)})
SAIDA = BASE / 'Base HHI'
resumo.write_csv(SAIDA / 'memoria_calculo_concentracao.csv')
ev.write_csv(SAIDA / 'concentracao_por_ano.csv')
print('\nExportado: memoria_calculo_concentracao.csv e concentracao_por_ano.csv')
print('\nConferencia esperada: 71.739 mercados | 90,2% alt.conc | 36,6% unico |')
print('  HHI mediano 7.803 | 84,6% (>=2) | 69,1% (>=5) | 53,7% (>=10) | 96,4% do valor')